In [22]:
import numpy as np
import tensorflow as tf
import pickle

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [23]:
corpus = [
    "I love learning machine learning",
    "I love learning artificial intelligence",
    "I enjoy learning machine learning",
    "I enjoy learning artificial intelligence",
    "machine learning is very interesting",
    "machine learning is very useful",
    "machine learning is a part of artificial intelligence",
    "deep learning is a part of machine learning",
    "deep learning is very powerful",
    "artificial intelligence is changing the world",
    "artificial intelligence is very useful",
    "artificial intelligence is growing rapidly",
    "data science is related to machine learning",
    "data science is very interesting",
    "python is useful for machine learning",
    "python is widely used in artificial intelligence",
    "neural networks are used in deep learning",
    "neural networks can learn complex patterns",
    "deep learning uses neural networks",
    "machine learning models learn from data",
    "machine learning models can make predictions",
    "artificial intelligence can solve complex problems",
    "deep learning can process large amounts of data",
    "data is important for machine learning",
    "good data helps machine learning models",
    "training a model requires data",
    "a neural network learns from examples",
    "the model learns patterns from data",
    "the model can make predictions",
    "learning from data is important",
    "machine learning is changing technology",
    "artificial intelligence is changing technology",
    "deep learning is changing technology",
    "python and machine learning are very useful",
    "python is easy to learn",
    "machine learning can be difficult to learn",
    "practice makes machine learning easier",
    "learning artificial intelligence requires practice",
    "learning machine learning requires practice",
    "deep learning requires a lot of data",
]

print("Number of sentences:", len(corpus))

Number of sentences: 40


In [24]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1
print("Vocab Size:", total_words)

Vocab Size: 67


In [25]:
print(tokenizer.word_index)

{'<OOV>': 1, 'learning': 2, 'is': 3, 'machine': 4, 'artificial': 5, 'intelligence': 6, 'data': 7, 'deep': 8, 'very': 9, 'can': 10, 'a': 11, 'i': 12, 'useful': 13, 'of': 14, 'changing': 15, 'python': 16, 'neural': 17, 'learn': 18, 'from': 19, 'requires': 20, 'the': 21, 'to': 22, 'networks': 23, 'models': 24, 'model': 25, 'technology': 26, 'practice': 27, 'love': 28, 'enjoy': 29, 'interesting': 30, 'part': 31, 'science': 32, 'for': 33, 'used': 34, 'in': 35, 'are': 36, 'complex': 37, 'patterns': 38, 'make': 39, 'predictions': 40, 'important': 41, 'learns': 42, 'powerful': 43, 'world': 44, 'growing': 45, 'rapidly': 46, 'related': 47, 'widely': 48, 'uses': 49, 'solve': 50, 'problems': 51, 'process': 52, 'large': 53, 'amounts': 54, 'good': 55, 'helps': 56, 'training': 57, 'network': 58, 'examples': 59, 'and': 60, 'easy': 61, 'be': 62, 'difficult': 63, 'makes': 64, 'easier': 65, 'lot': 66}


In [26]:
input_sequences = []
for sentence in corpus:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[: i + 1]
        input_sequences.append(n_gram_sequence)
        
print("Total sequences:", len(input_sequences))

Total sequences: 191


In [27]:
max_sequence_length = max(len(sequence) for sequence in input_sequences)
print("Max Seq Len:", max_sequence_length)

Max Seq Len: 8


In [28]:
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_length, padding="pre")
)
print("Input sequence shape:", input_sequences.shape)

Input sequence shape: (191, 8)


In [29]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (191, 7)
y shape: (191,)


In [30]:
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
print("Target shape:", y.shape)

Target shape: (191, 67)


In [31]:
model = Sequential(
    [
        Embedding(input_dim=total_words, output_dim=100),
        LSTM(150),
        Dense(total_words, activation="softmax"),
    ]
)

In [32]:
Embedding(input_dim=total_words, output_dim=100)

<Embedding name=embedding_3, built=False>

In [33]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [35]:
early_stopping = EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)

In [36]:
history = model.fit(
    X, y, epochs=200, batch_size=32, callbacks=[early_stopping], verbose=1
)

Epoch 1/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.1047 - loss: 4.1920 
Epoch 2/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.1466 - loss: 4.1164
Epoch 3/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1466 - loss: 3.8909
Epoch 4/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.1466 - loss: 3.6921
Epoch 5/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1204 - loss: 3.6293
Epoch 6/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.1309 - loss: 3.5895
Epoch 7/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1571 - loss: 3.5328
Epoch 8/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.1466 - loss: 3.4919
Epoch 9/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.1518 - loss: 3.4440
Epoch 10/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1675 - loss: 3.3925
Epoch 11/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.2356 - loss: 3.3174
Epoch 12/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1937 - l

In [37]:
model.save("model.keras")

In [38]:
with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)